In [ ]:
import xarray as xr
from pystac_client import Client
import lazycogs
import stac_geoparquet
import geopandas as gpd
import pandas as pd
import obstore
import pyproj
import numpy as np
from matplotlib import pyplot as plt

from pheno import double_logistic

In [ ]:
transition_model = pd.read_parquet("phenocam_phenology_model.parquet")
meta = pd.read_parquet("phenocam_site_in_usa.parquet")
site = "jasperridge"

target_lon = meta.loc["jasperridge", "lon"]
target_lat = meta.loc["jasperridge", "lat"]

transformer = pyproj.Transformer.from_crs(4326, 8857, always_xy=True)

search_bbox = (target_lon, target_lat, target_lon+1e-3, target_lat+1e-3)
target_bbox = transformer.transform_bounds(*search_bbox)

In [ ]:
x = np.linspace(-100, 365)
i = 0

beta_up = transition_model.iloc[0]["model_doy_transition_50_rising"]
beta_dn = beta_up + transition_model.iloc[0]["model_season_length"]
k_up    = transition_model.iloc[0]["transition_slope_rising"]
k_dn    = transition_model.iloc[0]["transition_slope_falling"]


plt.plot(x, double_logistic(x, k_up, k_dn, beta_up, beta_dn))

In [ ]:
csde_stac = Client.open("https://stac.dataspace.copernicus.eu/v1/")

lsp_items = csde_stac.search(
    collections="clms_ndvi_global_300m_10daily_v3_cog",
    bbox=search_bbox,
    datetime="2015-01-01/2022-01-01",
    limit=100
).item_collection().to_dict()

print(f"Found {len(lsp_items["features"])} features")

In [ ]:
# Sanitize items
for item in lsp_items["features"]:
    # Remove property keys that are not always present and cause failure on conversion
    # to geoparquet.
    item["properties"].pop("platform", None)
    item["properties"].pop("constellation", None)
    # Remove incorrect datetime properties
    item["properties"].pop("datetime", None)
    item["properties"].pop("start_datetime", None)
    item["properties"].pop("end_datetime", None)
    # Parse the correct datetime from the item ID
    dt_str = item["id"].split("_")[3]
    item["properties"]["datetime"] = pd.to_datetime(dt_str, utc=True)
    # Add asset-level proj code so lazycogs knows what's going on
    for asset_key in item["assets"]:
        item["assets"][asset_key]["proj:code"] = "EPSG:4326"

In [ ]:
stac_geoparquet.to_geodataframe(
    lsp_items["features"], 
    dtype_backend="numpy_nullable"
).to_parquet("/tmp/ndvi_items.parquet")

Set up S3 access

In [ ]:
import configparser

config = configparser.ConfigParser()
config.read("/home/jovyan/.s3cfg")

access_key = config.get("cdse", "access_key")
secret_key = config.get("cdse", "secret_key")

store = obstore.store.S3Store(
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    bucket="eodata",
    endpoint="https://eodata.dataspace.copernicus.eu"
)

Open the data array

In [ ]:
lsp_da = lazycogs.open(
    "/tmp/ndvi_items.parquet",
    bands=["ndvi"],
    store=store,
    crs="EPSG:8857",
    resolution=300,
    max_concurrent_reads=4,
    bbox=target_bbox
)

In [ ]:
scale = lsp_items["features"][0]["assets"]["ndvi"]["raster:scale"]
offset = lsp_items["features"][0]["assets"]["ndvi"]["raster:offset"]

site_ndvi_ts = (lsp_da.squeeze().compute() * scale) + offset

In [ ]:
# These CSVs say they only have NDVI but also have the GCC values we did modeling with
camera_gcc_ts = pd.read_csv("https://phenocam.nau.edu/data/archive/jasperridge/ROI/jasperridge_GR_2000_ndvi_1day.csv", skiprows=22)
camera_gcc_ts["date"] = pd.to_datetime(camera_gcc_ts["date"])

In [ ]:
transition_model.query("site == 'jasperridge' and roi_id == 2000")

In [ ]:
site_ndvi_ts

In [ ]:
from matplotlib import pyplot as plt

curves_to_model = transition_model.query("site == 'jasperridge' and roi_id == 2000")
fig, ax = plt.subplots(2, 3, sharex=False, sharey=False)

doy_x = np.linspace(-100, 365, num=100)

for (_, row), axis in zip(curves_to_model.iterrows(), ax.flat):
    # Determine what year we are modeling
    year = row["transition_50_rising"].year
    if row["doy_transition_50_rising"] < 0: 
        year += 1

    # NDVI time series
    site_ndvi_ts_subset = site_ndvi_ts.sel(time=slice(f"{year-1}-09-01", f"{year}-12-31")).copy()
    if site_ndvi_ts_subset.sizes["time"] > 0:
        # Normalize the signal
        site_ndvi_ts_subset = (site_ndvi_ts_subset - site_ndvi_ts_subset.min())  / (site_ndvi_ts_subset.max() - site_ndvi_ts_subset.min())
        site_ndvi_ts_subset.plot(x="time", ax=axis)

    # GCC time series
    camera_gcc_ts_subset = camera_gcc_ts.query(f"year == {year}").copy()
    camera_gcc_ts_subset["gcc_90"] = (camera_gcc_ts_subset["gcc_90"] - camera_gcc_ts_subset["gcc_90"].min()) / (camera_gcc_ts_subset["gcc_90"].max() - camera_gcc_ts_subset["gcc_90"].min())
    camera_gcc_ts_subset.plot(ax=axis, kind="line", x="date", y="gcc_90", legend=False)
    axis.set_title(year)

    # Modeled phenology time series
    beta_up = row["model_doy_transition_50_rising"]
    beta_dn = beta_up + row["model_season_length"]
    k_up    = row["transition_slope_rising"]
    k_dn    = row["transition_slope_falling"]

    sim_curve = double_logistic(doy_x, k_up, k_dn, beta_up, beta_dn)
    doy_x_datetime = pd.to_datetime(f"{year}-01-01") + pd.to_timedelta(doy_x, unit="days")
    axis.plot(doy_x_datetime, sim_curve)

plt.tight_layout()

#site_ndvi_ts.plot(ax=ax, label="CDSE NDVI")
#camera_gcc_ts.query("year == 2019").plot(ax=ax, kind="line", x="date", y="gcc_90", label="PhenoCam GCC90")
#plt.legend()
#plt.title("")